# 📊 NeMo Fraud Detection: Chat-Baseline-Evaluierung

Dieses Notebook führt eine faire Chat-Baseline-Evaluierung des unmodifizierten Basismodells (**Llama-3.1-8B-Instruct** via NVIDIA NIM) vor dem Supervised Fine-Tuning (SFT) durch.

### Funktionsumfang:
1. **Pre-Flight-Check:** Prüft vorab, ob der NIM-Service erreichbar ist (Fail-Fast).
2. **API-Abfrage:** Sendet validierte Transkripte per Chat-Completion-Endpunkt an das Modell.
3. **Label-Mapping & Parsing:** Mappt Ground-Truth-Labels und extrahiert die Modellantwort robust via Regex.
4. **Metriken & Tracking:** Berechnet die Accuracy und loggt die Ergebnisse an Weights & Biases (`wandb`).

In [7]:
import json
import re
import requests
import wandb
import sys

# Konfiguration der Pfade und Endpunkte
VAL_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft/validation.jsonl"
NIM_URL = "http://172.17.0.1:8800/v1/chat/completions"
MODEL_NAME = "meta/llama-3.1-8b-instruct"

print("✅ Bibliotheken und Konfigurationen geladen.")

✅ Bibliotheken und Konfigurationen geladen.


### 1. Pre-Flight-Check (Verbindungstest zum NIM-Server)

In [8]:
def check_llm_connection():
    """Prüft vor dem Start, ob der NIM-Service erreichbar ist."""
    print("🔍 Prüfe Verbindung zum LLM (NIM-Service)...")
    try:
        health_url = "http://172.17.0.1:8800/v1/models"
        response = requests.get(health_url, timeout=5)
        if response.status_code == 200:
            print("✅ Verbindung zum LLM steht!")
            return True
        else:
            print(f"❌ Verbindung fehlgeschlagen. Status-Code: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Verbindung zum NIM-Service unter '{NIM_URL}' nicht möglich: {e}")
        return False

# Verbindungstest direkt ausführen
if not check_llm_connection():
    raise SystemExit("Abbruch: LLM-Server ist nicht erreichbar.")

🔍 Prüfe Verbindung zum LLM (NIM-Service)...
✅ Verbindung zum LLM steht!


### 2. Durchführung der Chat-Baseline-Evaluierung

In [9]:
def run_chat_baseline():
    wandb.init(project="fraud-detection", name="baseline-chat-evaluation")
    print(f"🚀 Starte faire Chat-Baseline-Evaluierung...\n")
    
    correct = 0
    total = 0
    
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            
            input_content = data.get("input", "")
            raw_true = data.get("output", "").strip().lower()
            
            # Mappe deutsche Labels auf das englische Schema ('fraud' / 'legitimate')
            if "betrug" in raw_true and "kein" not in raw_true:
                true_label = "fraud"
            else:
                true_label = "legitimate"
            
            if not input_content:
                continue
                
            payload = {
                "model": MODEL_NAME,
                "messages": [
                    {
                        "role": "system",
                        "content": "Du bist ein Betrugserkennungs-Assistent für Kundengespräche. Analysiere das Transkript und antworte AUSSCHLIESSLICH mit exakt einem Wort: 'fraud' oder 'legitimate'."
                    },
                    {
                        "role": "user",
                        "content": input_content
                    }
                ],
                "temperature": 0.0,
                "max_tokens": 10
            }
            
            try:
                response = requests.post(NIM_URL, json=payload)
                result = response.json()
                
                if "choices" not in result:
                    print(f"Fehler von API bei Eintrag {total + 1}: {result}")
                    continue

                # Antwort aus der Chat-Struktur auslesen
                model_raw = result["choices"][0]["message"]["content"].strip().lower()
                
                # Robustes Parsing via Regex
                if "legitimate" in model_raw and "fraud" not in model_raw:
                    model_answer = "legitimate"
                elif "fraud" in model_raw and "legitimate" not in model_raw:
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_raw)
                    model_answer = match.group(1) if match else model_raw
                
                is_correct = (true_label == model_answer)
                if is_correct:
                    correct += 1
                total += 1
                
                print(f"Eintrag {total} | Erwartet: {true_label} | Erkannt: {model_answer} | Korrekt: {is_correct}")
                
            except Exception as e:
                print(f"Fehler bei Eintrag {total + 1}: {e}")

    accuracy = (correct / total) * 100 if total > 0 else 0
    
    # Metriken an wandb übertragen
    wandb.log({
        "eval/accuracy": accuracy,
        "eval/correct": correct,
        "eval/total": total
    })
    wandb.finish()

    print("\n" + "="*40)
    print("📊 CHAT-BASELINE EVALUIERUNGSERGEBNISSE")
    print("="*40)
    print(f"✅ Korrekt erkannt: {correct} / {total}")
    print(f"🎯 Genauigkeit:     {accuracy:.2f}%")
    print("="*40)

# Start der Evaluierung
run_chat_baseline()

🚀 Starte faire Chat-Baseline-Evaluierung...

Eintrag 1 | Erwartet: legitimate | Erkannt: legitimate | Korrekt: True
Eintrag 2 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 3 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 4 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 5 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 6 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 7 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 8 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 9 | Erwartet: legitimate | Erkannt: legitimate | Korrekt: True
Eintrag 10 | Erwartet: legitimate | Erkannt: legitimate | Korrekt: True
Eintrag 11 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 12 | Erwartet: legitimate | Erkannt: fraud | Korrekt: False
Eintrag 13 | Erwartet: legitimate | Erkannt: legitimate | Korrekt: True
Eintrag 14 | Erwartet: legitimate | Erkannt: legitimate | Korrek

eval/accuracy,▁
eval/correct,▁
eval/total,▁
eval/accuracy,48.37662
eval/correct,149
eval/total,308



📊 CHAT-BASELINE EVALUIERUNGSERGEBNISSE
✅ Korrekt erkannt: 149 / 308
🎯 Genauigkeit:     48.38%
